# Imports

In [1]:
from general_functions import compute_rot_ang
import plot_constants as pc
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import ListedColormap

# old: not sure whats actually needed here
import ast
from astropy.convolution import convolve_fft
from astropy.coordinates import SkyCoord
from astropy.nddata import Cutout2D
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
import astropy.units as u 
from astropy.wcs import WCS
import astropy.constants as const
from astropy.visualization import wcsaxes, simple_norm
import astropy.wcs as wcs
from astropy.wcs import WCS
from astropy.wcs.utils import proj_plane_pixel_area
import bdsf
from glob import glob
import matplotlib
import matplotlib.pyplot as plt
matplotlib.style.use('my_style.mplstyle')
import numpy as np
import os
import pandas as pd
import pyregion
from regions import Regions

import radio_beam
from radio_beam import Beam
from reproject import reproject_exact, reproject_interp
import scipy.ndimage as ndimage
from scipy.ndimage import label
from scipy import stats
from regions import Region, Regions
from photutils.aperture import EllipticalAperture, CircularAperture, RectangularAperture

import warnings
# Filter out the FITSFixedWarning
from astropy.wcs import FITSFixedWarning
warnings.filterwarnings('ignore', category=FITSFixedWarning, append=True)
import plot_constants as pc
import run_constants as rc
from general_functions import mask_stacked_cube
#warnings.filterwarnings('ignore', category=VerifyWarning, append=True)
tar_res_scale = 48 * u.arcsec # Exlunding NGC2820 and NGC3448 allowed us to enhance the resolution from 65 to 48 arcsec
max_ang_extent = rc.max_ang_extent_arcmin
cal_distance = rc.cal_distance_Mpc * u.Mpc
regrid_pix_scale_arcsec = rc.regrid_pix_scale_arcsec
regrid_npix = rc.regrid_npix
n_diam_ir = rc.n_diam_ir

# manual_masking = ['NGC2683', 'NGC3877', 'NGC4157', 'NGC5907'] # DArray (old)
manual_masking = ['NGC4302','NGC5907'] 
import pickle
with open('run_constants.pkl', 'rb') as f:
    run_const_dic = pickle.load(f)
    
with open("circ_ellipse_mask_dict.pkl", "rb") as f:
    circ_ellipse_mask_dict = pickle.load(f)

with open("PI_regrid_ang_noise_dict.pkl", "rb") as handle:
    noise_dict_PI_regrid_ang = pickle.load(handle)
    
with open("PI_raw_noise_dict.pkl", "rb") as handle:
    noise_dict_PI_raw = pickle.load(handle)     

with open('abs_rm_dict.pkl', 'rb') as f:
    abs_rm_dict = pickle.load(f)
    
def array_stats(a):
    a = a.ravel()
    n = len(a)
    mean = np.nanmean(a)
    std = np.nanstd(a)
    minimum = np.nanmin(a)
    maximum = np.nanmax(a)
    return f"N Elements={n}, Mean={mean}, Std={std}, Min={minimum}, Max={maximum}"    

high_res_sample = rc.high_res_sample
high_res_sample_ang = rc.high_res_sample_ang

stacks = [  "stack_ang_align_noNorm_mean",
            "stack_ang_align_noNorm_median",
            "stack_ang_align_piNorm_mean",
            "stack_ang_align_piNorm_mean_exclhighrm",
            "stack_ang_align_piNorm_median",
            "stack_ang_align_piNorm_median_exclhighrm",
            "stack_ang_align_piNorm_median_highrm",
            "stack_ang_align_piNorm_median_lowrm",
            "stack_ang_standard_noNorm_mean",
            "stack_ang_standard_noNorm_median",
            "stack_ang_standard_piNorm_mean",
            "stack_ang_standard_piNorm_mean_exclhighrm",
            "stack_ang_standard_piNorm_median",
            "stack_ang_standard_piNorm_median_exclhighrm",
            "stack_ang_both_noNorm_mean",
            "stack_ang_both_noNorm_median",
            "stack_ang_both_piNorm_mean",
            "stack_ang_both_piNorm_mean_exclhighrm",
            "stack_ang_both_piNorm_median",
            "stack_ang_both_piNorm_median_exclhighrm",
            "stack_ang_both_piNorm_median_highrm",
            "stack_ang_both_piNorm_median_lowrm",
            "stack_phy_align_distNorm_median",
            "stack_phy_align_piNorm_median",
            "stack_phy_align_piNorm_median_ang_sample",
            "stack_phy_both_distNorm_median",
            "stack_phy_both_piNorm_median",
            "stack_phy_standard_distNorm_median",
            "stack_phy_standard_piNorm_median"
]


In [ ]:
df = pd.read_csv('df_final.csv')
df['abs_rm_weighted_mean'] = df['galaxy'].map(abs_rm_dict)

display(df)
df_ang = pd.read_csv('df_ang.csv')
df_ang['abs_rm_weighted_mean'] = df_ang['galaxy'].map(abs_rm_dict)

df_phy = pd.read_csv('df_phy.csv')


# Galactic RM Correction

In [ ]:
def pol_ang(q, u, return_deg=False):
    """compute the polarisation angle from the Stokes parameters Q and U 

    Args:
        q (_type_): Stokes Q
        u (_type_): Stokes U

    Returns:
        _type_: polarisation angle
    """
    pa_rad = 0.5 * np.arctan2(u, q)
    if return_deg:
        pa_deg = np.degrees(pa_rad)
        return pa_deg
    else:
        return pa_rad

def pol_ang_source(pa_obs, freq, rm):
    wavelength = const.c / freq 
    pa_s = pa_obs - rm * wavelength**2
    return pa_s

def pol_int(q, u):
    """compute the linear polarised intensity from the Stokes parameters Q and U 

    Args:
        q (_type_): Stokes Q
        u (_type_): Stokes U

    Returns:
        _type_: polarised intensity
    """
    pi = np.sqrt(q**2 + u**2)
    return pi

def foreground_RM_correction(pol_ang_cube, freqs, rm):
    pol_ang_source_cube = np.zeros(shape=pol_ang_cube.shape)
    for i in range(0, len(freqs)):
        pol_ang = pol_ang_cube[i]
        wavelength = const.c.value / freqs[i]
        pol_ang_source = pol_ang - rm * wavelength**2
        pol_ang_source_cube[i] = pol_ang_source
    return pol_ang_source_cube

def q_source(pol_int, pol_ang_source):
    q = pol_int * np.cos(2*pol_ang_source)
    return q

def u_source(pol_int, pol_ang_source):
    u = pol_int * np.sin(2*pol_ang_source)
    return u



In [ ]:
QU_source_dir = 'cube_processing/qu_source'
if not os.path.exists(QU_source_dir):
    os.makedirs(QU_source_dir)
print("----- Galactic RM Foreground Correction -----")
for index, row in df.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    rm_mean = row['rm_mean']
    rm_std = row['rm_std']
    print(f"Working on galaxy: {gal}:")
    print(f"RM mean +/- std: {rm_mean} +/- {rm_std}")
    freq_file = row['freq_txt_C']
    freqs = np.loadtxt(freq_file)
    print(f"Number of frequencies in freq file: {len(freqs)}")
    central_freq = freqs[int(len(freqs)/2)]
    central_lambda = const.c.value / central_freq
    print(f"Central Wavelength [cm]: {central_lambda*100}")
    central_rot = rm_mean * central_lambda**2
    central_rot_deg = np.rad2deg(central_rot)
    print(f"Rotation at central frequency [rad,deg]: {central_rot, central_rot_deg}")
    q_cube = row['Q_fits_C']
    u_cube = row['U_fits_C']
    ff_Q = fits.open(q_cube)
    ff_Q.info()
    q_dat = ff_Q[0].data
    q_head = ff_Q[0].header
    w_cel = WCS(q_head).celestial
    cel_head = w_cel.to_header()
    ff_U = fits.open(u_cube)
    ff_U.info()
    u_dat = ff_U[0].data
    #for i in range(0, len(freqs)):
    pol_int_cube = pol_int(q_dat, u=u_dat)
    pol_ang_cube = pol_ang(q=q_dat, u=u_dat)
    fits.writeto(f"cube_processing/{gal}_pol_int_uncorrected.fits", data=pol_int_cube, overwrite=True, header=cel_head)
    fits.writeto(f"cube_processing/{gal}_pol_ang_uncorrected.fits", data=pol_ang_cube, overwrite=True, header=cel_head)
    pol_ang_source_cube = foreground_RM_correction(pol_ang_cube=pol_ang_cube, freqs=freqs, rm=rm_mean)
    print("---- Pol Ang Cube Statistics ----")
    print("Pol ang uncorrected (rad):")
    print(array_stats(pol_ang_cube))
    print("Pol ang corrected (rad):")
    print(array_stats(pol_ang_source_cube))
    print("---------------------------------") 
    fits.writeto(f"cube_processing/{gal}_pol_ang_corrected.fits", data=pol_ang_source_cube, overwrite=True, header=cel_head)
    q_source_cube = q_source(pol_int=pol_int_cube, pol_ang_source=pol_ang_source_cube)
    u_source_cube = u_source(pol_int=pol_int_cube, pol_ang_source=pol_ang_source_cube)
    pi_source_cube = pol_int(q=q_source_cube, u=u_source_cube) #np.sqrt(q_source_cube**2 + u_source_cube**2)
    pa_source_cube = pol_ang(q=q_source_cube, u=u_source_cube)
    fits.writeto(f"cube_processing/qu_source/{gal}_q_source.fits", data=q_source_cube, overwrite=True, header=cel_head) 
    fits.writeto(f"cube_processing/qu_source/{gal}_u_source.fits", data=u_source_cube, overwrite=True, header=cel_head)
    fits.writeto(f"cube_processing/qu_source/{gal}_pi_source.fits", data=pi_source_cube, overwrite=True, header=cel_head)  
    fits.writeto(f"cube_processing/qu_source/{gal}_pa_source.fits", data=pa_source_cube, overwrite=True, header=cel_head)  

    print("Done with this galaxy")
    
    

# Galaxy Alignment

In [ ]:
def create_circular_mask(array):
    # Get the dimensions of the array
    h, w = array.shape
    
    # Find the center of the array
    center = (int(h / 2), int(w / 2))
    
    # Create a grid of distances from the center
    Y, X = np.ogrid[:h, :w]
    dist_from_center = np.sqrt((X - center[1])**2 + (Y - center[0])**2)
    
    # Initialize the maximum radius
    max_radius = min(center[0], center[1], h - center[0], w - center[1])
    
    # Find the maximum radius that does not include any NaN values
    for radius in range(max_radius, 0, -1):
        mask = dist_from_center <= radius
        if not np.any(np.isnan(array[mask])):
            break
    
    # Create the final mask
    final_mask = dist_from_center <= radius
    
    return final_mask

def rotate_cubes(Q_cube, U_cube, RotMatrix):
    # Get the dimensions of the cubes
    freq_dim, x_dim, y_dim = Q_cube.shape
    
    # Create empty cubes for the rotated values
    Q_prime_cube = np.zeros_like(Q_cube)
    U_prime_cube = np.zeros_like(U_cube)
    
    # Iterate through each pixel in the cubes
    for f in range(freq_dim):
        for x in range(x_dim):
            for y in range(y_dim):
                # Extract the Q and U values at the current pixel
                Q = Q_cube[f, x, y]
                U = U_cube[f, x, y]
                
                # Create the vector
                vector = np.array([Q, U])
                
                # Perform the rotation using matrix multiplication
                rotated_vector = np.dot(RotMatrix, vector)
                
                # Extract the rotated components
                Q_prime, U_prime = rotated_vector
                
                # Save the rotated values in the new cubes
                Q_prime_cube[f, x, y] = Q_prime
                U_prime_cube[f, x, y] = U_prime
    
    return Q_prime_cube, U_prime_cube

def rotate_cubes_new(pi_source_cube, pa_source_cube, rot_ang, return_checks=False):
    rot_ang_rad = np.radians(rot_ang)
    print(f"Rotation in rad: {rot_ang_rad}")
    # Get the dimensions of the cubes
    freq_dim, x_dim, y_dim = pi_source_cube.shape
    # Create empty cubes for the rotated values
    #Q_prime_cube = np.zeros_like(pi_source_cube)
    #U_prime_cube = np.zeros_like(pi_source_cube)
    #pa_prime_cube = np.zeros_like(pi_source_cube)

    pa_prime_cube = pa_source_cube - rot_ang_rad
    Q_prime_cube = pi_source_cube * np.cos(2*pa_prime_cube)
    U_prime_cube = pi_source_cube * np.sin(2*pa_prime_cube)
    Q_prime_cube_fill = np.nan_to_num(Q_prime_cube)
    U_prime_cube_fill = np.nan_to_num(U_prime_cube)


    Q_prime_rot_cube = ndimage.rotate(Q_prime_cube_fill, angle=rot_ang, axes=(1,2), reshape=False)
    U_prime_rot_cube = ndimage.rotate(U_prime_cube_fill, angle=rot_ang, axes=(1,2), reshape=False)
    if return_checks:
        return Q_prime_rot_cube, U_prime_rot_cube, Q_prime_cube, U_prime_cube, pa_prime_cube
    else:
        return Q_prime_rot_cube, U_prime_rot_cube


        

In [ ]:
print("----- Galaxy Alignment -----")
for index, row in df.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    print(f"Working on galaxy: {gal}:")
    pos_ang = row['Hyperleda_pa[deg]']
    rot_ang_fine_tuning = row['PA_fine_tunig_hyperleda_ref']
    #pi = fits.getdata(filename=f'cube_processing/{gal}_pol_int_uncorrected.fits')
    
    q_source = fits.getdata(filename=f'cube_processing/qu_source/{gal}_q_source.fits')
    u_source = fits.getdata(filename=f'cube_processing/qu_source/{gal}_u_source.fits')
    pa_source  = fits.getdata(filename=f'cube_processing/qu_source/{gal}_pa_source.fits')
    pi_source = fits.getdata(filename=f'cube_processing/qu_source/{gal}_pi_source.fits')

    rot_ang = compute_rot_ang(pos_ang, rot_ang_fine_tuning)
    q_prime_rot, u_prime_rot, q_prime, u_prime, pa_prime = rotate_cubes_new(pi_source_cube=pi_source,
                                                                            pa_source_cube=pa_source,
                                                                            rot_ang=rot_ang,
                                                                            return_checks=True)
    fits.writeto(f"cube_processing/qu_rot/{gal}_q_rot.fits", data=q_prime_rot, overwrite=True)
    fits.writeto(f"cube_processing/qu_rot/{gal}_u_rot.fits", data=u_prime_rot, overwrite=True)
    fits.writeto(f"cube_processing/qu_rot/{gal}_q_prime_notrot.fits", data=q_prime, overwrite=True)
    fits.writeto(f"cube_processing/qu_rot/{gal}_u_prime_notrot.fits", data=u_prime, overwrite=True)
    fits.writeto(f"cube_processing/qu_rot/{gal}_pa_prime_notrot.fits", data=pa_prime, overwrite=True)
    
    # if row['extra_rot']=='yes':
    #     print("Additional 180 deg rotation required for aligning rotation sense:")
    #     rot_ang_align = rot_ang + 180
    #     q_prime_rot, u_prime_rot, q_prime, u_prime, pa_prime = rotate_cubes_new(pi_source_cube=pi_source,
    #                                                                         pa_source_cube=pa_source,
    #                                                                         rot_ang=rot_ang_align,
    #                                                                         return_checks=True)
    #     fits.writeto(f"cube_processing/qu_rot/{gal}_q_rot_align.fits", data=q_prime_rot, overwrite=True)
    #     fits.writeto(f"cube_processing/qu_rot/{gal}_u_rot_align.fits", data=u_prime_rot, overwrite=True)
    # Produce two data sets per galaxy (rotated by 180 deg).
    print("Additional 180 deg rotation required for aligning rotation sense:")
    rot_ang_align = rot_ang + 180
    q_prime_rot, u_prime_rot, q_prime, u_prime, pa_prime = rotate_cubes_new(pi_source_cube=pi_source,
                                                                        pa_source_cube=pa_source,
                                                                        rot_ang=rot_ang_align,
                                                                        return_checks=True)
    fits.writeto(f"cube_processing/qu_rot/{gal}_q_rot_extra180.fits", data=q_prime_rot, overwrite=True)
    fits.writeto(f"cube_processing/qu_rot/{gal}_u_rot_extra180.fits", data=u_prime_rot, overwrite=True)

        
    print("Done with this galaxy")

# Masking Back- and Foreground Sources

In [ ]:
use_circ_ellipse_mask = True
print("----- Masking -----")
for index, row in df.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    print(f"Working on galaxy: {gal}:")
    # if row['extra_rot']=="yes":
    #     q_files = [f'cube_processing/qu_rot/{gal}_q_rot.fits',f'cube_processing/qu_rot/{gal}_q_rot_align.fits']
    #     u_files = [f'cube_processing/qu_rot/{gal}_u_rot.fits',f'cube_processing/qu_rot/{gal}_u_rot_align.fits']
    #     output_tags = ["mask.fits", "align_mask.fits"]
    # else:
    #     q_files = [f'cube_processing/qu_rot/{gal}_q_rot.fits']
    #     u_files = [f'cube_processing/qu_rot/{gal}_u_rot.fits']
    #     output_tags = ["mask.fits"]
    # print(q_files, u_files, output_tags)
    q_files = [f'cube_processing/qu_rot/{gal}_q_rot.fits',f'cube_processing/qu_rot/{gal}_q_rot_extra180.fits']
    u_files = [f'cube_processing/qu_rot/{gal}_u_rot.fits',f'cube_processing/qu_rot/{gal}_u_rot_extra180.fits']
    output_tags = ["mask.fits", "extra180_mask.fits"]
    for j in np.arange(0,len(q_files)):
        print(f"Running Index: {j}")
        q_rot = fits.getdata(filename=q_files[j])
        u_rot = fits.getdata(filename=u_files[j])
        mask = fits.getdata(filename=f'geom_rotate_mask/out_fits/masks/{gal}_rot_mask_final.fits')
        
        if output_tags[j] == "extra180_mask.fits":
            mask = np.array([row[::-1] for row in mask[::-1]]) # Turn Mask by 180 degrees for the rotated realisations
            
        q_spec, q_x, q_y = q_rot.shape
        u_spec, u_x, u_y = q_rot.shape
        mask_x, mask_y = mask.shape
        print (f"Shape Q-Cube: {q_rot.shape}")
        print (f"Shape U-Cube: {u_rot.shape}")
        print(f"Shape Mask: {mask.shape}")
        if use_circ_ellipse_mask:
            center = (mask_x// 2, mask_y // 2)
            print("Also Apply a circular/elliptical mask to mask out regions close to the edge of the primary beam.")
            gal_id = f"n{gal[3:]}"
            mask_prop = circ_ellipse_mask_dict[gal_id]
            mask_type = mask_prop[0]
            mask_params = mask_prop[1]
            if mask_type == "ell":
                a, b, theta = mask_params
                aperture = EllipticalAperture(center, a/2, b/2, theta)
            elif mask_type == "cir":
                d = mask_params[0]
                aperture = CircularAperture(center, d/2)
            print(aperture)
            cir_ellipse_mask = aperture.to_mask(method='center')
            cir_ellipse_mask_array = cir_ellipse_mask.to_image(shape=mask.shape)
            cir_ellipse_mask_array = np.where(cir_ellipse_mask_array==0, np.nan, cir_ellipse_mask_array)

        q_masked = np.zeros(shape=q_rot.shape)
        u_masked = np.zeros(shape=q_rot.shape)

        for i in range(q_spec):
            q_masked[i,:,:]= q_rot[i,:,:] * mask
            if use_circ_ellipse_mask:
                q_masked[i,:,:]=q_masked[i,:,:]*cir_ellipse_mask_array            
        
        for i in range(u_spec):
            u_masked[i,:,:]= u_rot[i,:,:] * mask
            if use_circ_ellipse_mask:
                u_masked[i,:,:]=u_masked[i,:,:]*cir_ellipse_mask_array
        # Getting the Header information from the original fits file (image has only been rotated yet) change CRVAL1/2 to 0, 0 
        ## Q   
        original_q_head = fits.getheader(row['Q_fits_C'])
        beam_q = Beam(major=original_q_head['BMAJ']*u.deg, minor=original_q_head['BMIN']*u.deg, pa=original_q_head['BPA']*u.deg)
        w_q = WCS(original_q_head)
        new_q_head=w_q.to_header()
        new_q_head['BUNIT'] = original_q_head['BUNIT']
        new_q_head['CRVAL1']=0
        new_q_head['CRVAl2']=0
        new_q_head.update(beam_q.to_header_keywords())
        ## U
        original_u_head = fits.getheader(row['U_fits_C'])
        beam_u = Beam(major=original_u_head['BMAJ']*u.deg, minor=original_u_head['BMIN']*u.deg, pa=original_u_head['BPA']*u.deg)
        w_u = WCS(original_u_head)
        new_u_head=w_u.to_header()
        new_u_head['BUNIT'] = original_u_head['BUNIT']
        new_u_head['CRVAL1']=0
        new_u_head['CRVAl2']=0
        new_u_head.update(beam_u.to_header_keywords())
        pi_masked = np.sqrt(q_masked**2 + u_masked**2)
        # Write output images 
        fits.writeto(filename=f"cube_processing/qu_mask/{gal}_q_{output_tags[j]}", data=q_masked, header=new_q_head, overwrite=True)
        fits.writeto(filename=f"cube_processing/qu_mask/{gal}_u_{output_tags[j]}", data=u_masked, header=new_u_head, overwrite=True)
        fits.writeto(filename=f"cube_processing/qu_mask/{gal}_pi_{output_tags[j]}", data=pi_masked, header=new_u_head, overwrite=True)
    print("Done with this galaxy")

# Convolution

## Angular

In [ ]:
print("----- Convolution (Angular Scaling) -----")
df_ang = pd.read_csv('df_ang.csv')
for index, row in df_ang.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    print(f"Working on galaxy: {gal}:")
    # if row['extra_rot']=="yes":
    #     q_files = [f'cube_processing/qu_mask/{gal}_q_mask.fits',f'cube_processing/qu_mask/{gal}_q_align_mask.fits']
    #     u_files = [f'cube_processing/qu_mask/{gal}_u_mask.fits',f'cube_processing/qu_mask/{gal}_u_align_mask.fits']
    #     output_tags = ["conv_ang.fits", "align_conv_ang.fits"]
    # else:
    #     q_files = [f'cube_processing/qu_mask/{gal}_q_mask.fits']
    #     u_files = [f'cube_processing/qu_mask/{gal}_u_mask.fits']
    #     output_tags = ["conv_ang.fits"]
        
    
    q_files = [f'cube_processing/qu_mask/{gal}_q_mask.fits',f'cube_processing/qu_mask/{gal}_q_extra180_mask.fits']
    u_files = [f'cube_processing/qu_mask/{gal}_u_mask.fits',f'cube_processing/qu_mask/{gal}_u_extra180_mask.fits']
    output_tags = ["conv_ang.fits", "extra180_conv_ang.fits"]    
    print(q_files, u_files, output_tags)
    for j in np.arange(0,len(q_files)):
        print(f"Running Index: {j}")
        q_fits = fits.open(q_files[j])
        u_fits = fits.open(u_files[j])
        q_mask = q_fits[0].data
        u_mask = u_fits[0].data
        q_head, u_head = q_fits[0].header,  u_fits[0].header
        beam_q = Beam.from_fits_header(q_head)
        beam_u = Beam.from_fits_header(u_head)
        # Print the beam parameters in arcseconds and degrees
        print(f"Q-Beam: BMAJ={beam_q.major.to(u.arcsec):.2f}, BMIN={beam_q.minor.to(u.arcsec):.2f}, PA={beam_q.pa.to(u.deg):.2f}")
        print(f"U-Beam: BMAJ={beam_u.major.to(u.arcsec):.2f}, BMIN={beam_u.minor.to(u.arcsec):.2f}, PA={beam_u.pa.to(u.deg):.2f}")

        tar_bmaj_arcsec= row['target_res_angular']
        tar_beam = radio_beam.Beam(major=tar_bmaj_arcsec*u.arcsec)
        # Q and U cubes have the same beam! Therefore, we only use the Q beam from now on.
        print(f"Target Beam: BMAJ={tar_beam.major.to(u.arcsec):.2f}, BMIN={tar_beam.minor.to(u.arcsec):.2f}, PA={tar_beam.pa.to(u.deg):.2f}")
        deconv_beam = tar_beam.deconvolve(beam_q)
        beam_axis_prod_tar = tar_beam.major.to(u.arcsec) * tar_beam.minor.to(u.arcsec)
        beam_axis_prod_cur = beam_q.major.to(u.arcsec) * beam_q.minor.to(u.arcsec)
        beam_factor = (beam_axis_prod_tar / beam_axis_prod_cur)
        print('Beam Factor:', beam_factor.value)
        beam_factor = beam_factor.value
        print(f"Deconvolution Beam: BMAJ={deconv_beam.major.to(u.arcsec):.2f}, BMIN={deconv_beam.minor.to(u.arcsec):.2f}, PA={deconv_beam.pa.to(u.deg):.2f}")
        w_cubes = WCS(q_head).celestial
        pixel_scale = wcs.utils.proj_plane_pixel_scales(w_cubes)
        print(f"Pixel Scale [arcsec]: {pixel_scale*3600}")
        kernel_scale=pixel_scale[0]*u.degree
        deconv_kernel = deconv_beam.as_kernel(pixscale=kernel_scale)
        # Updating headers:
        q_head['BMAJ']=tar_beam.major.to(u.degree).value
        q_head['BMIN']=tar_beam.minor.to(u.degree).value
        q_head['BPA']=tar_beam.pa.to(u.deg).value
        u_head['BMAJ']=tar_beam.major.to(u.degree).value
        u_head['BMIN']=tar_beam.minor.to(u.degree).value
        u_head['BPA']=tar_beam.pa.to(u.deg).value
        q_conv = np.zeros_like(q_mask)
        u_conv = np.zeros_like(u_mask)
        print("Convolving cubes")
        for masked, convolved in zip([q_mask, u_mask], [q_conv,u_conv]):
            f, _, _ = masked.shape
            for i in range(f):
                cube_slice = masked[i, :, :]
                cube_slice_nans = np.where(np.isnan(cube_slice), np.nan, 1.)
                convolved[i, :, :]= convolve_fft(cube_slice, kernel=deconv_kernel)
                convolved[i, :, :] = convolved[i, :, :] * cube_slice_nans
        q_conv = q_conv * beam_factor
        u_conv = u_conv * beam_factor
        fits.writeto(filename=f"cube_processing/qu_conv_ang/{gal}_q_{output_tags[j]}", data=q_conv, header=q_head, overwrite=True)
        fits.writeto(filename=f"cube_processing/qu_conv_ang/{gal}_u_{output_tags[j]}", data= u_conv, header=u_head, overwrite=True)
    print("Done with this galaxy")

### high resolution run

In [ ]:
print("----- Convolution (Angular Scaling) -----")
df_ang = pd.read_csv('df_ang.csv')
for index, row in df_ang.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    if not gal in high_res_sample_ang:
        print(f"Skipping galaxy {gal} as it is not in the high-res sample.")
        continue
    print(f"Working on galaxy: {gal}:")
    # if row['extra_rot']=="yes":
    #     q_files = [f'cube_processing/qu_mask/{gal}_q_mask.fits',f'cube_processing/qu_mask/{gal}_q_align_mask.fits']
    #     u_files = [f'cube_processing/qu_mask/{gal}_u_mask.fits',f'cube_processing/qu_mask/{gal}_u_align_mask.fits']
    #     output_tags = ["conv_ang.fits", "align_conv_ang.fits"]
    # else:
    #     q_files = [f'cube_processing/qu_mask/{gal}_q_mask.fits']
    #     u_files = [f'cube_processing/qu_mask/{gal}_u_mask.fits']
    #     output_tags = ["conv_ang.fits"]
        
    
    q_files = [f'cube_processing/qu_mask/{gal}_q_mask.fits',f'cube_processing/qu_mask/{gal}_q_extra180_mask.fits']
    u_files = [f'cube_processing/qu_mask/{gal}_u_mask.fits',f'cube_processing/qu_mask/{gal}_u_extra180_mask.fits']
    output_tags = ["conv_ang.fits", "extra180_conv_ang.fits"]    
    print(q_files, u_files, output_tags)
    for j in np.arange(0,len(q_files)):
        print(f"Running Index: {j}")
        q_fits = fits.open(q_files[j])
        u_fits = fits.open(u_files[j])
        q_mask = q_fits[0].data
        u_mask = u_fits[0].data
        q_head, u_head = q_fits[0].header,  u_fits[0].header
        beam_q = Beam.from_fits_header(q_head)
        beam_u = Beam.from_fits_header(u_head)
        # Print the beam parameters in arcseconds and degrees
        print(f"Q-Beam: BMAJ={beam_q.major.to(u.arcsec):.2f}, BMIN={beam_q.minor.to(u.arcsec):.2f}, PA={beam_q.pa.to(u.deg):.2f}")
        print(f"U-Beam: BMAJ={beam_u.major.to(u.arcsec):.2f}, BMIN={beam_u.minor.to(u.arcsec):.2f}, PA={beam_u.pa.to(u.deg):.2f}")

        tar_bmaj_arcsec= row['target_res_angular_highres']
        tar_beam = radio_beam.Beam(major=tar_bmaj_arcsec*u.arcsec)
        # Q and U cubes have the same beam! Therefore, we only use the Q beam from now on.
        print(f"Target Beam: BMAJ={tar_beam.major.to(u.arcsec):.2f}, BMIN={tar_beam.minor.to(u.arcsec):.2f}, PA={tar_beam.pa.to(u.deg):.2f}")
        deconv_beam = tar_beam.deconvolve(beam_q)
        beam_axis_prod_tar = tar_beam.major.to(u.arcsec) * tar_beam.minor.to(u.arcsec)
        beam_axis_prod_cur = beam_q.major.to(u.arcsec) * beam_q.minor.to(u.arcsec)
        beam_factor = (beam_axis_prod_tar / beam_axis_prod_cur)
        print('Beam Factor:', beam_factor.value)
        beam_factor = beam_factor.value
        print(f"Deconvolution Beam: BMAJ={deconv_beam.major.to(u.arcsec):.2f}, BMIN={deconv_beam.minor.to(u.arcsec):.2f}, PA={deconv_beam.pa.to(u.deg):.2f}")
        w_cubes = WCS(q_head).celestial
        pixel_scale = wcs.utils.proj_plane_pixel_scales(w_cubes)
        print(f"Pixel Scale [arcsec]: {pixel_scale*3600}")
        kernel_scale=pixel_scale[0]*u.degree
        deconv_kernel = deconv_beam.as_kernel(pixscale=kernel_scale)
        # Updating headers:
        q_head['BMAJ']=tar_beam.major.to(u.degree).value
        q_head['BMIN']=tar_beam.minor.to(u.degree).value
        q_head['BPA']=tar_beam.pa.to(u.deg).value
        u_head['BMAJ']=tar_beam.major.to(u.degree).value
        u_head['BMIN']=tar_beam.minor.to(u.degree).value
        u_head['BPA']=tar_beam.pa.to(u.deg).value
        q_conv = np.zeros_like(q_mask)
        u_conv = np.zeros_like(u_mask)
        print("Convolving cubes")
        for masked, convolved in zip([q_mask, u_mask], [q_conv,u_conv]):
            f, _, _ = masked.shape
            for i in range(f):
                cube_slice = masked[i, :, :]
                cube_slice_nans = np.where(np.isnan(cube_slice), np.nan, 1.)
                convolved[i, :, :]= convolve_fft(cube_slice, kernel=deconv_kernel)
                convolved[i, :, :] = convolved[i, :, :] * cube_slice_nans
        q_conv = q_conv * beam_factor
        u_conv = u_conv * beam_factor
        fits.writeto(filename=f"cube_processing/qu_conv_ang_highres/{gal}_q_{output_tags[j]}", data=q_conv, header=q_head, overwrite=True)
        fits.writeto(filename=f"cube_processing/qu_conv_ang_highres/{gal}_u_{output_tags[j]}", data= u_conv, header=u_head, overwrite=True)
    print("Done with this galaxy")

## Physical

Rewrite fits files with haeder and beam in kpc:

In [ ]:
print("----- Prep Convolution (Physical Scaling) -----")
df_phy = pd.read_csv('df_phy.csv')
for index, row in df_phy.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    dist_Mpc = row['d[Mpc]']
    print(f"Working on galaxy: {gal}:")
    print(f"Distance [Mpc]: {dist_Mpc}")
    kpc_per_arcsec = dist_Mpc * 1000 * np.tan(np.radians(1 / 3600))
    print(f"kpc per arcsec at given distance: {kpc_per_arcsec}")
    q_files = [f'cube_processing/qu_mask/{gal}_q_mask.fits',f'cube_processing/qu_mask/{gal}_q_extra180_mask.fits']
    u_files = [f'cube_processing/qu_mask/{gal}_u_mask.fits',f'cube_processing/qu_mask/{gal}_u_extra180_mask.fits']
    output_tags = ["conv_phy_prep.fits", "extra180_conv_phy_prep.fits"]    
    print(q_files, u_files, output_tags)
    for j in np.arange(0,len(q_files)):
        print(f"Running Index: {j}")
        q_fits = fits.open(q_files[j])
        u_fits = fits.open(u_files[j])
        q_mask = q_fits[0].data
        u_mask = u_fits[0].data
        q_head, u_head = q_fits[0].header,  u_fits[0].header
        print(f"Q Pix scale [arcsec]: {np.array([q_head['CDELT1'], q_head['CDELT2']])*3600}")
        q_pix_kpc= np.array([q_head['CDELT1'], q_head['CDELT2']])*3600*(kpc_per_arcsec)
        print(f"Q Pix scale [kpc]: {q_pix_kpc}")
        beam_q = Beam.from_fits_header(q_head)
        beam_u = Beam.from_fits_header(u_head)
        # Print the beam parameters in arcseconds and degrees
        print(f"Q-Beam: BMAJ={beam_q.major.to(u.arcsec):.2f}, BMIN={beam_q.minor.to(u.arcsec):.2f}, PA={beam_q.pa.to(u.deg):.2f}")
        print(f"U-Beam: BMAJ={beam_u.major.to(u.arcsec):.2f}, BMIN={beam_u.minor.to(u.arcsec):.2f}, PA={beam_u.pa.to(u.deg):.2f}")
        
        for head_item in ["CDELT1", "CDELT2", "BMAJ", "BMIN"]:
            q_head[head_item] = q_head[head_item] *3600*(kpc_per_arcsec)
            u_head[head_item] = u_head[head_item] *3600*(kpc_per_arcsec)
        q_head["CUNIT1"] = "kpc"
        q_head["CUNIT2"] = "kpc"
        u_head["CUNIT1"] = "kpc"
        u_head["CUNIT2"] = "kpc"
        q_head["CTYPE1"] = "X"
        q_head["CTYPE2"] = "Y"
        u_head["CTYPE1"] = "X"
        u_head["CTYPE2"] = "Y"
        fits.writeto(filename=f"cube_processing/qu_conv_phy_prep/{gal}_q_{output_tags[j]}", data=q_mask, header=q_head, overwrite=True)
        fits.writeto(filename=f"cube_processing/qu_conv_phy_prep/{gal}_u_{output_tags[j]}", data= u_mask, header=u_head, overwrite=True)
    print("Done with this galaxy")

In [ ]:
print("----- Convolution (Physical Scaling Scaling) -----")
df_phy = pd.read_csv('df_phy.csv')
tar_res_phys_kpc = 2.0
for index, row in df_phy.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    print(f"Working on galaxy: {gal}:")    
    q_files = [f'cube_processing/qu_conv_phy_prep/{gal}_q_conv_phy_prep.fits',f'cube_processing/qu_conv_phy_prep/{gal}_q_extra180_conv_phy_prep.fits']
    u_files = [f'cube_processing/qu_conv_phy_prep/{gal}_u_conv_phy_prep.fits',f'cube_processing/qu_conv_phy_prep/{gal}_u_extra180_conv_phy_prep.fits']
    output_tags = ["conv_phy.fits", "extra180_conv_phy.fits"]    
    print(q_files, u_files, output_tags)
    for j in np.arange(0,len(q_files)):
        print(f"Running Index: {j}")
        q_fits = fits.open(q_files[j])
        u_fits = fits.open(u_files[j])
        q_mask = q_fits[0].data
        u_mask = u_fits[0].data
        q_head, u_head = q_fits[0].header,  u_fits[0].header
        beam_q = Beam.from_fits_header(q_head)
        beam_u = Beam.from_fits_header(u_head)
        # Print the beam parameters in arcseconds and degrees
        # computing new beam by manually
        tar_bmaj_kpc = rc.phy_res_kpc
        cur_bmaj_kpc = q_head["BMAJ"]
        conv_bmaj_kpc = np.sqrt(tar_bmaj_kpc**2 - cur_bmaj_kpc**2)
        beam_factor_kpc = tar_bmaj_kpc**2/cur_bmaj_kpc**2
        print(f"Beam FWHMs in kpc:")
        print(f"Target Resolution: {tar_bmaj_kpc} [kpc]")
        print(f"Current Resolution: {cur_bmaj_kpc} [kpc]")
        print(f"Beam for Convolution: {conv_bmaj_kpc} [kpc]")
        print(f"Beam Factor for Jy/Beam: {beam_factor_kpc}")
        print(f"Pixel Scale [kpc]: {q_head["CDELT1"]}, {q_head["CDELT2"]}")
        conv_bmaj_pix = conv_bmaj_kpc/np.abs(q_head["CDELT1"])
        print(f"Beam for Convolution: {conv_bmaj_pix} [pix]")
        conv_bmaj_pix_stdev = conv_bmaj_pix/(2*np.sqrt(2*np.log(2)))
        print(f"Beam for Convolution (stdev): {conv_bmaj_pix_stdev} [pix]")
        conv_kernel = radio_beam.EllipticalGaussian2DKernel(stddev_maj=conv_bmaj_pix_stdev,
                                                            stddev_min=conv_bmaj_pix_stdev,
                                                            position_angle=0)
        fits.writeto(filename="conv_kernel_manual.fits", data=conv_kernel.array, overwrite=True)
        # Updating headers:
        q_head['BMAJ']=2.0
        q_head['BMIN']=2.0
        q_head['BPA']=0
        u_head['BMAJ']=2.0
        u_head['BMIN']=2.0
        u_head['BPA']=0
        q_conv = np.zeros_like(q_mask)
        u_conv = np.zeros_like(u_mask)
        print("Convolving cubes")
        for masked, convolved in zip([q_mask, u_mask], [q_conv,u_conv]):
            f, _, _ = masked.shape # type: ignore
            for i in range(f):
                cube_slice = masked[i, :, :]
                cube_slice_nans = np.where(np.isnan(cube_slice), np.nan, 1.)
                convolved[i, :, :]= convolve_fft(cube_slice, kernel=conv_kernel, normalize_kernel=True)
                convolved[i, :, :] = convolved[i, :, :] * cube_slice_nans
        q_conv = q_conv * beam_factor_kpc
        u_conv = u_conv * beam_factor_kpc
        fits.writeto(filename=f"cube_processing/qu_conv_phy/{gal}_q_{output_tags[j]}", data=q_conv, header=q_head, overwrite=True)
        fits.writeto(filename=f"cube_processing/qu_conv_phy/{gal}_u_{output_tags[j]}", data= u_conv, header=u_head, overwrite=True)
    print("Done with this galaxy")

### high resolution run

In [ ]:
print("----- Convolution (Physical Scaling Scaling) -----")
df_phy = pd.read_csv('df_phy.csv')
tar_res_phys_kpc = 1.03
for index, row in df_phy.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    if not gal in high_res_sample:
        print(f"Skipping galaxy {gal} as it is not in the high-res sample.")
        continue
    print(f"Working on galaxy: {gal}:")    
    q_files = [f'cube_processing/qu_conv_phy_prep/{gal}_q_conv_phy_prep.fits',f'cube_processing/qu_conv_phy_prep/{gal}_q_extra180_conv_phy_prep.fits']
    u_files = [f'cube_processing/qu_conv_phy_prep/{gal}_u_conv_phy_prep.fits',f'cube_processing/qu_conv_phy_prep/{gal}_u_extra180_conv_phy_prep.fits']
    output_tags = ["conv_phy.fits", "extra180_conv_phy.fits"]    
    print(q_files, u_files, output_tags)
    for j in np.arange(0,len(q_files)):
        print(f"Running Index: {j}")
        q_fits = fits.open(q_files[j])
        u_fits = fits.open(u_files[j])
        q_mask = q_fits[0].data
        u_mask = u_fits[0].data
        q_head, u_head = q_fits[0].header,  u_fits[0].header
        beam_q = Beam.from_fits_header(q_head)
        beam_u = Beam.from_fits_header(u_head)
        # Print the beam parameters in arcseconds and degrees
        # computing new beam by manually
        tar_bmaj_kpc = tar_res_phys_kpc
        cur_bmaj_kpc = q_head["BMAJ"]
        conv_bmaj_kpc = np.sqrt(tar_bmaj_kpc**2 - cur_bmaj_kpc**2)
        beam_factor_kpc = tar_bmaj_kpc**2/cur_bmaj_kpc**2
        print(f"Beam FWHMs in kpc:")
        print(f"Target Resolution: {tar_bmaj_kpc} [kpc]")
        print(f"Current Resolution: {cur_bmaj_kpc} [kpc]")
        print(f"Beam for Convolution: {conv_bmaj_kpc} [kpc]")
        print(f"Beam Factor for Jy/Beam: {beam_factor_kpc}")
        print(f"Pixel Scale [kpc]: {q_head["CDELT1"]}, {q_head["CDELT2"]}")
        conv_bmaj_pix = conv_bmaj_kpc/np.abs(q_head["CDELT1"])
        print(f"Beam for Convolution: {conv_bmaj_pix} [pix]")
        conv_bmaj_pix_stdev = conv_bmaj_pix/(2*np.sqrt(2*np.log(2)))
        print(f"Beam for Convolution (stdev): {conv_bmaj_pix_stdev} [pix]")
        conv_kernel = radio_beam.EllipticalGaussian2DKernel(stddev_maj=conv_bmaj_pix_stdev,
                                                            stddev_min=conv_bmaj_pix_stdev,
                                                            position_angle=0)
        fits.writeto(filename="conv_kernel_manual.fits", data=conv_kernel.array, overwrite=True)
        # Updating headers:
        q_head['BMAJ']=2.0
        q_head['BMIN']=2.0
        q_head['BPA']=0
        u_head['BMAJ']=2.0
        u_head['BMIN']=2.0
        u_head['BPA']=0
        q_conv = np.zeros_like(q_mask)
        u_conv = np.zeros_like(u_mask)
        print("Convolving cubes")
        for masked, convolved in zip([q_mask, u_mask], [q_conv,u_conv]):
            f, _, _ = masked.shape # type: ignore
            for i in range(f):
                cube_slice = masked[i, :, :]
                cube_slice_nans = np.where(np.isnan(cube_slice), np.nan, 1.)
                convolved[i, :, :]= convolve_fft(cube_slice, kernel=conv_kernel, normalize_kernel=True)
                convolved[i, :, :] = convolved[i, :, :] * cube_slice_nans
        q_conv = q_conv * beam_factor_kpc
        u_conv = u_conv * beam_factor_kpc
        fits.writeto(filename=f"cube_processing/qu_conv_phy_highres/{gal}_q_{output_tags[j]}", data=q_conv, header=q_head, overwrite=True)
        fits.writeto(filename=f"cube_processing/qu_conv_phy_highres/{gal}_u_{output_tags[j]}", data= u_conv, header=u_head, overwrite=True)
    print("Done with this galaxy")

# Regrid

## Angular

In [ ]:
print("----- Regrid (Angular Scaling) -----")
for index, row in df_ang.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    print(f"Working on galaxy: {gal}:")
    # if row['extra_rot']=="yes":
    #     q_files = [f'cube_processing/qu_conv_ang/{gal}_q_conv_ang.fits',f'cube_processing/qu_conv_ang/{gal}_q_align_conv_ang.fits']
    #     u_files = [f'cube_processing/qu_conv_ang/{gal}_u_conv_ang.fits',f'cube_processing/qu_conv_ang/{gal}_u_align_conv_ang.fits']
    #     output_tags = ["regrid_ang.fits", "align_regrid_ang.fits"]
    # else:
    #     q_files = [f'cube_processing/qu_conv_ang/{gal}_q_conv_ang.fits']
    #     u_files = [f'cube_processing/qu_conv_ang/{gal}_u_conv_ang.fits']
    #     output_tags = ["regrid_ang.fits"]
    q_files = [f'cube_processing/qu_conv_ang/{gal}_q_conv_ang.fits',f'cube_processing/qu_conv_ang/{gal}_q_extra180_conv_ang.fits']
    u_files = [f'cube_processing/qu_conv_ang/{gal}_u_conv_ang.fits',f'cube_processing/qu_conv_ang/{gal}_u_extra180_conv_ang.fits']
    output_tags = ["regrid_ang.fits", "extra180_regrid_ang.fits"]
    print(q_files, u_files, output_tags)
    for j in np.arange(0,len(q_files)):
        print(f"Running Index: {j}")
        q_fits = fits.open(q_files[j])
        u_fits = fits.open(u_files[j])
        q_conv = q_fits[0].data
        u_conv = u_fits[0].data
        q_head, u_head = q_fits[0].header,  u_fits[0].header
        beam_q = Beam.from_fits_header(q_head)
        beam_u = Beam.from_fits_header(u_head)
        # Print the beam parameters in arcseconds and degrees
        print(f"Q-Beam: BMAJ={beam_q.major.to(u.arcsec):.2f}, BMIN={beam_q.minor.to(u.arcsec):.2f}, PA={beam_q.pa.to(u.deg):.2f}")
        print(f"U-Beam: BMAJ={beam_u.major.to(u.arcsec):.2f}, BMIN={beam_u.minor.to(u.arcsec):.2f}, PA={beam_u.pa.to(u.deg):.2f}")
        w_cubes = WCS(q_head).celestial
        print("Current WCS:")
        print(w_cubes)
        pixel_scale = wcs.utils.proj_plane_pixel_scales(w_cubes)
        print(f"Current pixel scale [arcsec]: {pixel_scale*3600}")
        new_pix_size = beam_q.major.to(u.arcsec)/run_const_dic['beam_pix_ratio_angular']
        new_pix_size_deg = new_pix_size.to(u.deg)
        print(f"New pixel scale for regridding (1/5 * BMAJ): {new_pix_size}")
        #tar_wcs = w_cubes.copy()
        new_npix= int(rc.regrid_npix_ang)
        tar_wcs = wcs.WCS()
        tar_wcs.wcs.ctype = ['RA---SIN', 'DEC--SIN']
        tar_wcs.wcs.cdelt = np.array([-new_pix_size_deg.value,new_pix_size_deg.value])
        tar_wcs.wcs.crpix = np.array([int(new_npix/2),int(new_npix/2)])
        print("Target WCS:")
        print(tar_wcs)
        f, _, _ = q_conv.shape
        q_regrid = np.zeros((f, new_npix, new_npix))
        u_regrid = np.zeros((f, new_npix, new_npix))
        print(f"Starting to regrid individual slices to new shape [{new_npix},{new_npix}]:")
        for i in range(f):
            # Q
            q_slice = q_conv[i,:,:]
            q_slice_regrid = reproject_exact(input_data=(q_slice, w_cubes),
                                            output_projection=tar_wcs, shape_out=[new_npix, new_npix],
                                            return_footprint=False, parallel=6)
            q_regrid[i, :, :] = q_slice_regrid
            # U 
            u_slice = u_conv[i,:,:]
            u_slice_regrid = reproject_exact(input_data=(u_slice, w_cubes),
                                            output_projection=tar_wcs, shape_out=[new_npix, new_npix],
                                            return_footprint=False, parallel=6)
            u_regrid[i, :, :] = u_slice_regrid
        
        q_head.update(tar_wcs.to_header())
        u_head.update(tar_wcs.to_header())
        fits.writeto(filename=f"cube_processing/qu_regrid_ang/{gal}_q_{output_tags[j]}", data=q_regrid, header=q_head, overwrite=True)
        fits.writeto(filename=f"cube_processing/qu_regrid_ang/{gal}_u_{output_tags[j]}", data= u_regrid, header=u_head, overwrite=True)
    print("Done with this galaxy")

### high resolution run

In [ ]:
print("----- Regrid (Angular Scaling) -----")
for index, row in df_ang.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    if not gal in high_res_sample_ang:
        print(f"Skipping galaxy {gal} as it is not in the high-res sample.")
        continue
    print(f"Working on galaxy: {gal}:")
    # if row['extra_rot']=="yes":
    #     q_files = [f'cube_processing/qu_conv_ang/{gal}_q_conv_ang.fits',f'cube_processing/qu_conv_ang/{gal}_q_align_conv_ang.fits']
    #     u_files = [f'cube_processing/qu_conv_ang/{gal}_u_conv_ang.fits',f'cube_processing/qu_conv_ang/{gal}_u_align_conv_ang.fits']
    #     output_tags = ["regrid_ang.fits", "align_regrid_ang.fits"]
    # else:
    #     q_files = [f'cube_processing/qu_conv_ang/{gal}_q_conv_ang.fits']
    #     u_files = [f'cube_processing/qu_conv_ang/{gal}_u_conv_ang.fits']
    #     output_tags = ["regrid_ang.fits"]
    q_files = [f'cube_processing/qu_conv_ang_highres/{gal}_q_conv_ang.fits',f'cube_processing/qu_conv_ang_highres/{gal}_q_extra180_conv_ang.fits']
    u_files = [f'cube_processing/qu_conv_ang_highres/{gal}_u_conv_ang.fits',f'cube_processing/qu_conv_ang_highres/{gal}_u_extra180_conv_ang.fits']
    output_tags = ["regrid_ang.fits", "extra180_regrid_ang.fits"]
    print(q_files, u_files, output_tags)
    for j in np.arange(0,len(q_files)):
        print(f"Running Index: {j}")
        q_fits = fits.open(q_files[j])
        u_fits = fits.open(u_files[j])
        q_conv = q_fits[0].data
        u_conv = u_fits[0].data
        q_head, u_head = q_fits[0].header,  u_fits[0].header
        beam_q = Beam.from_fits_header(q_head)
        beam_u = Beam.from_fits_header(u_head)
        # Print the beam parameters in arcseconds and degrees
        print(f"Q-Beam: BMAJ={beam_q.major.to(u.arcsec):.2f}, BMIN={beam_q.minor.to(u.arcsec):.2f}, PA={beam_q.pa.to(u.deg):.2f}")
        print(f"U-Beam: BMAJ={beam_u.major.to(u.arcsec):.2f}, BMIN={beam_u.minor.to(u.arcsec):.2f}, PA={beam_u.pa.to(u.deg):.2f}")
        w_cubes = WCS(q_head).celestial
        print("Current WCS:")
        print(w_cubes)
        pixel_scale = wcs.utils.proj_plane_pixel_scales(w_cubes)
        print(f"Current pixel scale [arcsec]: {pixel_scale*3600}")
        new_pix_size = beam_q.major.to(u.arcsec)/rc.ang_stack_pix_per_beam_hr
        new_pix_size_deg = new_pix_size.to(u.deg)
        print(f"New pixel scale for regridding (1/5 * BMAJ): {new_pix_size}")
        #tar_wcs = w_cubes.copy()
        new_npix= int(rc.regrid_npix_ang_hr)
        tar_wcs = wcs.WCS()
        tar_wcs.wcs.ctype = ['RA---SIN', 'DEC--SIN']
        tar_wcs.wcs.cdelt = np.array([-new_pix_size_deg.value,new_pix_size_deg.value])
        tar_wcs.wcs.crpix = np.array([int(new_npix/2),int(new_npix/2)])
        print("Target WCS:")
        print(tar_wcs)
        f, _, _ = q_conv.shape
        q_regrid = np.zeros((f, new_npix, new_npix))
        u_regrid = np.zeros((f, new_npix, new_npix))
        print(f"Starting to regrid individual slices to new shape [{new_npix},{new_npix}]:")
        for i in range(f):
            # Q
            q_slice = q_conv[i,:,:]
            q_slice_regrid = reproject_exact(input_data=(q_slice, w_cubes),
                                            output_projection=tar_wcs, shape_out=[new_npix, new_npix],
                                            return_footprint=False, parallel=6)
            q_regrid[i, :, :] = q_slice_regrid
            # U 
            u_slice = u_conv[i,:,:]
            u_slice_regrid = reproject_exact(input_data=(u_slice, w_cubes),
                                            output_projection=tar_wcs, shape_out=[new_npix, new_npix],
                                            return_footprint=False, parallel=6)
            u_regrid[i, :, :] = u_slice_regrid
        
        q_head.update(tar_wcs.to_header())
        u_head.update(tar_wcs.to_header())
        fits.writeto(filename=f"cube_processing/qu_regrid_ang_highres/{gal}_q_{output_tags[j]}", data=q_regrid, header=q_head, overwrite=True)
        fits.writeto(filename=f"cube_processing/qu_regrid_ang_highres/{gal}_u_{output_tags[j]}", data= u_regrid, header=u_head, overwrite=True)
    print("Done with this galaxy")

## Physical

In [ ]:
print("----- Regrid (Physical Scaling) -----")
for index, row in df_phy.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    print(f"Working on galaxy: {gal}:")
    q_files = [f'cube_processing/qu_conv_phy/{gal}_q_conv_phy.fits',f'cube_processing/qu_conv_phy/{gal}_q_extra180_conv_phy.fits']
    u_files = [f'cube_processing/qu_conv_phy/{gal}_u_conv_phy.fits',f'cube_processing/qu_conv_phy/{gal}_u_extra180_conv_phy.fits']
    output_tags = ["regrid_phy.fits", "extra180_regrid_phy.fits"]
    print(q_files, u_files, output_tags)
    pix_size_kpc = rc.phy_pix_size_kpc
    npix_tar = rc.n_pix_phy_ref_frame
    tar_wcs = wcs.WCS()
    tar_wcs.wcs.ctype = ['X', 'Y']
    tar_wcs.wcs.cdelt = np.array([-pix_size_kpc,pix_size_kpc])
    tar_wcs.wcs.crpix = np.array([int(npix_tar/2),int(npix_tar/2)])
    print("Target WCS:")
    print(tar_wcs)
    
    for j in np.arange(0,len(q_files)):
        print(f"Running Index: {j}")
        q_fits = fits.open(q_files[j])
        u_fits = fits.open(u_files[j])
        q_conv = q_fits[0].data
        u_conv = u_fits[0].data
        q_head, u_head = q_fits[0].header,  u_fits[0].header
        # Print the beam parameters in arcseconds and degrees
        cur_wcs = wcs.WCS()
        cur_wcs.wcs.ctype = ['X', 'Y']
        cur_wcs.wcs.cdelt = np.array([q_head["CDELT1"],q_head["CDELT2"]])
        cur_wcs.wcs.crpix = np.array([q_head["CRPIX1"],q_head["CRPIX2"]])
        print("Current WCS:")
        print(cur_wcs)
        
        f, _, _ = q_conv.shape
        q_regrid = np.zeros((f, npix_tar, npix_tar))
        u_regrid = np.zeros((f, npix_tar, npix_tar))
        print(f"Starting to regrid individual slices to new shape [{npix_tar},{npix_tar}]:")
        for i in range(f):
            # Q
            q_slice = q_conv[i,:,:]
            q_slice_regrid = reproject_interp(input_data=(q_slice, cur_wcs),
                                            output_projection=tar_wcs, shape_out=[npix_tar, npix_tar],
                                            return_footprint=False, parallel=6)
            q_regrid[i, :, :] = q_slice_regrid
            # U 
            u_slice = u_conv[i,:,:]
            u_slice_regrid = reproject_interp(input_data=(u_slice, cur_wcs),
                                            output_projection=tar_wcs, shape_out=[npix_tar, npix_tar],
                                            return_footprint=False, parallel=6)
            u_regrid[i, :, :] = u_slice_regrid
        
        q_head.update(tar_wcs.to_header())
        u_head.update(tar_wcs.to_header())
        fits.writeto(filename=f"cube_processing/qu_regrid_phy/{gal}_q_{output_tags[j]}", data=q_regrid, header=q_head, overwrite=True)
        fits.writeto(filename=f"cube_processing/qu_regrid_phy/{gal}_u_{output_tags[j]}", data= u_regrid, header=u_head, overwrite=True)
    print("Done with this galaxy")

### high resolution run

In [ ]:
print("----- Regrid (Physical Scaling) -----")
for index, row in df_phy.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    if not gal in high_res_sample:
        print(f"Skipping galaxy {gal} as it is not in the high-res sample.")
        continue
    print(f"Working on galaxy: {gal}:")
    q_files = [f'cube_processing/qu_conv_phy_highres/{gal}_q_conv_phy.fits',f'cube_processing/qu_conv_phy_highres/{gal}_q_extra180_conv_phy.fits']
    u_files = [f'cube_processing/qu_conv_phy_highres/{gal}_u_conv_phy.fits',f'cube_processing/qu_conv_phy_highres/{gal}_u_extra180_conv_phy.fits']
    output_tags = ["regrid_phy.fits", "extra180_regrid_phy.fits"]
    print(q_files, u_files, output_tags)
    pix_size_kpc = rc.phy_pix_size_kpc_hr
    npix_tar = rc.n_pix_phy_ref_frame_hr
    tar_wcs = wcs.WCS()
    tar_wcs.wcs.ctype = ['X', 'Y']
    tar_wcs.wcs.cdelt = np.array([-pix_size_kpc,pix_size_kpc])
    tar_wcs.wcs.crpix = np.array([int(npix_tar/2),int(npix_tar/2)])
    print("Target WCS:")
    print(tar_wcs)
    
    for j in np.arange(0,len(q_files)):
        print(f"Running Index: {j}")
        q_fits = fits.open(q_files[j])
        u_fits = fits.open(u_files[j])
        q_conv = q_fits[0].data
        u_conv = u_fits[0].data
        q_head, u_head = q_fits[0].header,  u_fits[0].header
        # Print the beam parameters in arcseconds and degrees
        cur_wcs = wcs.WCS()
        cur_wcs.wcs.ctype = ['X', 'Y']
        cur_wcs.wcs.cdelt = np.array([q_head["CDELT1"],q_head["CDELT2"]])
        cur_wcs.wcs.crpix = np.array([q_head["CRPIX1"],q_head["CRPIX2"]])
        print("Current WCS:")
        print(cur_wcs)
        
        f, _, _ = q_conv.shape
        q_regrid = np.zeros((f, npix_tar, npix_tar))
        u_regrid = np.zeros((f, npix_tar, npix_tar))
        print(f"Starting to regrid individual slices to new shape [{npix_tar},{npix_tar}]:")
        for i in range(f):
            # Q
            q_slice = q_conv[i,:,:]
            q_slice_regrid = reproject_interp(input_data=(q_slice, cur_wcs),
                                            output_projection=tar_wcs, shape_out=[npix_tar, npix_tar],
                                            return_footprint=False, parallel=6)
            q_regrid[i, :, :] = q_slice_regrid
            # U 
            u_slice = u_conv[i,:,:]
            u_slice_regrid = reproject_interp(input_data=(u_slice, cur_wcs),
                                            output_projection=tar_wcs, shape_out=[npix_tar, npix_tar],
                                            return_footprint=False, parallel=6)
            u_regrid[i, :, :] = u_slice_regrid
        
        q_head.update(tar_wcs.to_header())
        u_head.update(tar_wcs.to_header())
        fits.writeto(filename=f"cube_processing/qu_regrid_phy_highres/{gal}_q_{output_tags[j]}", data=q_regrid, header=q_head, overwrite=True)
        fits.writeto(filename=f"cube_processing/qu_regrid_phy_highres/{gal}_u_{output_tags[j]}", data= u_regrid, header=u_head, overwrite=True)
    print("Done with this galaxy")

# Additional Data Editing (Not main Routine)

## Check if CRPIX are always pointing to the central pixel

In [ ]:
for index, row in df.iterrows():
    gal = row['galaxy'].replace(" ","")
    print("----------")
    print(gal)
    head = fits.getheader(f"cube_processing/qu_source/{gal}_q_source.fits")
    w = WCS(head).celestial
    print(w)
    crpix1, crpix2 = head["CRPIX1"], head["CRPIX2"]
    skycoord_crpix = w.pixel_to_world(crpix1,crpix2)
    print(f"SkyCoord of reference Pixel: {skycoord_crpix}")
    sky_coord_table_string = f"{row['RA']} {row['Dec']}"
    skycoord_table = SkyCoord(ra=row['RA'], dec=row['Dec'], frame='icrs').transform_to('fk5')
    print(f"SkyCoord of of Galaxy in Table: {skycoord_table}")
    sep = skycoord_crpix.separation(skycoord_table).to(u.arcsec)
    print(f"Separation between Reference pixel and Table coordinate: {sep}")    
    print("----------")


## Mask RM Maps

 This can only run after rm_synth has been applied to the regridded cubes.

### Galaxies (Raw)

In [ ]:
print("----- Masking RM Maps based on PI Maps -----")
for index, row in df.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    print(f"Working on galaxy: {gal}:")
    gal_id = gal[3:]
    mean, std = noise_dict_PI_raw[f"n{gal_id}"]
    pi = fits.getdata(f'/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/n{gal_id}_raw/pf_n{gal_id}_raw_ampPeakPIfitEff.fits')
    pi_mask = np.where(pi>(mean+3*std), 1. , np.nan)
    rm_fits = fits.open(f'/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/n{gal_id}_raw/pf_n{gal_id}_raw_phiPeakPIfit_rm2.fits')
    rm_head = rm_fits[0].header
    rm_head['COMMENT']="Masked with PI Map (mean+3sig)"
    rm_dat = rm_fits[0].data
    rm_data_masked = rm_dat*pi_mask
    out_string = f'/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/n{gal_id}_raw/pf_n{gal_id}_raw_phiPeakPIfit_rm2_pi_masked.fits'
    print(f"Writing masked RM Map: {out_string}")
    fits.writeto(filename=out_string,
                data=rm_data_masked, header=rm_head, overwrite=True)
    
    
    
  
    

### Galaxies (Regrid Ang)

In [ ]:
  
print("----- Masking RM Maps based on PI Maps -----")
for index, row in df_ang.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    print(f"Working on galaxy: {gal}:")
    gal_id = gal[3:]
    mean, std = noise_dict_PI_regrid_ang[f"n{gal_id}"]
    pi = fits.getdata(f'/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/n{gal_id}_regrid_ang/pf_n{gal_id}_regrid_ang_ampPeakPIfitEff.fits')
    pi_mask = np.where(pi>(mean+3*std), 1. , np.nan)
    rm_fits = fits.open(f'/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/n{gal_id}_regrid_ang/pf_n{gal_id}_regrid_ang_phiPeakPIfit_rm2.fits')
    rm_head = rm_fits[0].header
    rm_wcs = WCS(rm_head).celestial
    rm_head['COMMENT']="Masked with PI Map (mean+3sig)"
    rm_dat = rm_fits[0].data
    rm_data_masked = rm_dat*pi_mask
    
                # --- REGION MASKING ---
    region_path = f"pi_apertures_flux_measurement_regrid_ang/n{gal_id}_PI.reg"
    print(f"open ds9 regions for masking: {region_path}")
    region = Regions.read(region_path, format='ds9')[0]
    print(region)
    print(rm_wcs)
    # Convert SkyRegion to PixelRegion using the WCS of the pi_map
    pixel_region = region.to_pixel(rm_wcs)
    shape = rm_dat.shape
    mask_obj = pixel_region.to_mask(mode='center')
    region_mask = mask_obj.to_image(shape)
    # region_mask is 1 inside, 0 outside, nan where undefined
    # Mask pi, rm, rm_err
    rm_data_masked = np.where(region_mask, rm_data_masked, np.nan)
    out_string = f'/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/n{gal_id}_regrid_ang/pf_n{gal_id}_regrid_ang_phiPeakPIfit_rm2_pi_masked.fits'
    print(f"Writing masked RM Map: {out_string}")
    fits.writeto(filename=out_string,
                data=rm_data_masked, header=rm_head, overwrite=True)
    
    
    if row['extra_rot']=="yes":
        if os.path.exists(f'/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/n{gal_id}_regrid_ang_align/pf_n{gal_id}_regrid_ang_align_ampPeakPIfitEff.fits'):
            pi = fits.getdata(f'/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/n{gal_id}_regrid_ang_align/pf_n{gal_id}_regrid_ang_align_ampPeakPIfitEff.fits')
            pi_mask = np.where(pi>(mean+3*std), 1. , np.nan)
            rm_fits = fits.open(f'/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/n{gal_id}_regrid_ang_align/pf_n{gal_id}_regrid_ang_align_phiPeakPIfit_rm2.fits')
            rm_head = rm_fits[0].header
            rm_head['COMMENT']="Masked with PI Map (mean+3sig)"
            rm_dat = rm_fits[0].data
            rm_data_masked = rm_dat*pi_mask
            fits.writeto(filename=f'/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/n{gal_id}_regrid_ang_align/pf_n{gal_id}_regrid_ang_align_phiPeakPIfit_rm2_pi_masked.fits',
                data=rm_data_masked, header=rm_head, overwrite=True)

### individual datasets:

In [ ]:
mean, std = noise_dict_PI_regrid_ang['n891']
mask_stacked_cube(rm_map="cube_processing/rm_analysis/3d_cubes/n891_regrid_ang_mirrored/pf_n891_regrid_ang_mirrored_phiPeakPIfit_rm2.fits",
                  pi_map="cube_processing/rm_analysis/3d_cubes/n891_regrid_ang_mirrored/pf_n891_regrid_ang_mirrored_ampPeakPIfitEff.fits",
                  pi_bkg_mean=mean, pi_bkg_std=std,
                  outname="cube_processing/rm_analysis/3d_cubes/n891_regrid_ang_mirrored/pf_n891_regrid_ang_mirrored_phiPeakPIfit_rm2_pi_masked.fits",
                  use_elliptical_mask=False)


mean, std = noise_dict_PI_regrid_ang['n891']
mask_stacked_cube(rm_map="cube_processing/rm_analysis/3d_cubes/n891_regrid_ang_extra180/pf_n891_regrid_ang_extra180_phiPeakPIfit_rm2.fits",
                  pi_map="cube_processing/rm_analysis/3d_cubes/n891_regrid_ang_extra180/pf_n891_regrid_ang_extra180_ampPeakPIfitEff.fits",
                  pi_bkg_mean=mean, pi_bkg_std=std,
                  outname="cube_processing/rm_analysis/3d_cubes/n891_regrid_ang_extra180/pf_n891_regrid_ang_extra180_phiPeakPIfit_rm2_pi_masked.fits",
                  use_elliptical_mask=False)

## Mirroring  Q and U Cubes

In [ ]:
def mirror_qu(q_path, u_path):
    q_fits, u_fits = fits.open(q_path), fits.open(u_path)
    q_dat, q_head = q_fits[0].data, q_fits[0].header
    u_dat, u_head = u_fits[0].data, u_fits[0].header
    print(u_dat.shape, q_dat.shape)
    q_flip = np.flip(q_dat, axis=2)
    # For Stokes-U, the sign of the pixel values need to change for the mirroring.
    u_flip = np.flip(u_dat, axis=2)*(-1.)
    q_head['Comment']='Data set mirrored with respect to the galaxy minor axis'
    u_head['Comment']='Data set mirrored with respect to the galaxy minor axis'
    q_out = q_path.split(".")[0] + "_mirrored.fits"
    u_out = u_path.split(".")[0] + "_mirrored.fits"
    fits.writeto(filename=q_out, data=q_flip, header=q_head, overwrite=True)
    fits.writeto(filename=u_out, data=u_flip, header=u_head, overwrite=True)

In [ ]:
print("----- Flip along galaxy minor axis (Angular Scaling) -----")
for index, row in df_ang.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    print(f"Working on galaxy: {gal}:")
    # if row['extra_rot']=="yes":
    #     q_files = [f'cube_processing/qu_conv_ang/{gal}_q_conv_ang.fits',f'cube_processing/qu_conv_ang/{gal}_q_align_conv_ang.fits']
    #     u_files = [f'cube_processing/qu_conv_ang/{gal}_u_conv_ang.fits',f'cube_processing/qu_conv_ang/{gal}_u_align_conv_ang.fits']
    #     output_tags = ["regrid_ang.fits", "align_regrid_ang.fits"]
    # else:
    #     q_files = [f'cube_processing/qu_conv_ang/{gal}_q_conv_ang.fits']
    #     u_files = [f'cube_processing/qu_conv_ang/{gal}_u_conv_ang.fits']
    #     output_tags = ["regrid_ang.fits"]
    mirror_qu(q_path=f'cube_processing/qu_regrid_ang/{gal}_q_regrid_ang.fits',
              u_path=f'cube_processing/qu_regrid_ang/{gal}_u_regrid_ang.fits')
    # output_tags = ["regrid_ang.fits", "extra180_regrid_ang.fits"]
    # print(q_files, u_files, output_tags)
    # for j in np.arange(0,len(q_files)):
    #     print(f"Running Index: {j}")
    #     q_fits = fits.open(q_files[j])
    #     u_fits = fits.open(u_files[j])
    #     q_conv = q_fits[0].data
    #     u_conv = u_fits[0].data
    #     q_head, u_head = q_fits[0].header,  u_fits[0].header
    #     beam_q = Beam.from_fits_header(q_head)
    #     beam_u = Beam.from_fits_header(u_head)
    #     # Print the beam parameters in arcseconds and degrees
    #     print(f"Q-Beam: BMAJ={beam_q.major.to(u.arcsec):.2f}, BMIN={beam_q.minor.to(u.arcsec):.2f}, PA={beam_q.pa.to(u.deg):.2f}")
    #     print(f"U-Beam: BMAJ={beam_u.major.to(u.arcsec):.2f}, BMIN={beam_u.minor.to(u.arcsec):.2f}, PA={beam_u.pa.to(u.deg):.2f}")
    #     w_cubes = WCS(q_head).celestial
    #     print("Current WCS:")
    #     print(w_cubes)
    #     pixel_scale = wcs.utils.proj_plane_pixel_scales(w_cubes)
    #     print(f"Current pixel scale [arcsec]: {pixel_scale*3600}")
    #     new_pix_size = beam_q.major.to(u.arcsec)/run_const_dic['beam_pix_ratio_angular']
    #     new_pix_size_deg = new_pix_size.to(u.deg)
    #     print(f"New pixel scale for regridding (1/5 * BMAJ): {new_pix_size}")
    #     #tar_wcs = w_cubes.copy()
    #     new_npix= int(run_const_dic['beam_pix_ratio_angular']*run_const_dic['min_sampling_ang']*2)
    #     tar_wcs = wcs.WCS()
    #     #ar_wcs.wcs.naxis = np.array([new_npix,new_npix])
    #     tar_wcs.wcs.ctype = ['RA---SIN', 'DEC--SIN']
    #     tar_wcs.wcs.cdelt = np.array([-new_pix_size_deg.value,new_pix_size_deg.value])
    #     tar_wcs.wcs.crpix = np.array([int(new_npix/2),int(new_npix/2)])
    #     print("Target WCS:")
    #     print(tar_wcs)
    #     f, _, _ = q_conv.shape
    #     q_regrid = np.zeros((f, new_npix, new_npix))
    #     u_regrid = np.zeros((f, new_npix, new_npix))
    #     print(f"Starting to regrid individual slices to new shape [{new_npix},{new_npix}]:")
    #     for i in range(f):
    #         # Q
    #         q_slice = q_conv[i,:,:]
    #         q_slice_regrid = reproject_exact(input_data=(q_slice, w_cubes),
    #                                         output_projection=tar_wcs, shape_out=[new_npix, new_npix],
    #                                         return_footprint=False, parallel=6)
    #         q_regrid[i, :, :] = q_slice_regrid
    #         # U 
    #         u_slice = u_conv[i,:,:]
    #         u_slice_regrid = reproject_exact(input_data=(u_slice, w_cubes),
    #                                         output_projection=tar_wcs, shape_out=[new_npix, new_npix],
    #                                         return_footprint=False, parallel=6)
    #         u_regrid[i, :, :] = u_slice_regrid
        
    #     q_head.update(tar_wcs.to_header())
    #     u_head.update(tar_wcs.to_header())
    #     fits.writeto(filename=f"cube_processing/qu_regrid_ang/{gal}_q_{output_tags[j]}", data=q_regrid, header=q_head, overwrite=True)
    #     fits.writeto(filename=f"cube_processing/qu_regrid_ang/{gal}_u_{output_tags[j]}", data= u_regrid, header=u_head, overwrite=True)
    print("Done with this galaxy")

## Extracting RM values from individual RM Maps

#### Individual Galaxies

In [ ]:
print("----- Extract RM values from individual galaxies -----")
rows = []

for index, row in df_ang.iterrows():
    print("-----")
    gal = row['galaxy'].replace(' ', '')
    gal_id = f"n{gal[3:]}"
    print(f"Working on galaxy: {gal}:")
    
    rm_map = fits.getdata(f"/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/{gal_id}_regrid_ang/pf_{gal_id}_regrid_ang_phiPeakPIfit_rm2_pi_masked.fits")
    rm_err_map = fits.getdata(f"/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/{gal_id}_regrid_ang/pf_{gal_id}_regrid_ang_dPhiPeakPIfit_rm2.fits")
    height, width = rm_map.shape
    center_x, center_y = width / 2, height / 2
    for y in range(height):
        for x in range(width):
            if not np.isnan(rm_map[y, x]):
                delta_x, delta_y = x - center_x, y - center_y
                rm = rm_map[y, x]
                rm_err = rm_err_map[y, x]
                row_dict = {"gal":gal, "x":delta_x, "y":delta_y, "rm":rm, "rm_err":rm_err}
                rows.append(row_dict)        

    #
    print("Done with this galaxy")
df_rm = pd.DataFrame(rows, columns=['gal', 'x', 'y', 'rm', 'rm_err'])
with open('df_rm_ang.pkl', 'wb') as f:
    pickle.dump(df_rm, f)

#### Stacks

In [2]:
print("----- Extract RM values from individual stacks -----")


rows=[]
for s in stacks:
    print(f"Working on stack: {s}:")
    rm_map = fits.getdata(f"/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/{s}/pf_{s}_phiPeakPIfit_rm2_pi_masked.fits")
    rm_err_map = fits.getdata(f"/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/{s}/pf_{s}_dPhiPeakPIfit_rm2.fits")
    height, width = rm_map.shape
    center_x, center_y = width / 2, height / 2
    for y in range(height):
        for x in range(width):
            if not np.isnan(rm_map[y, x]):
                delta_x, delta_y = x - center_x, y - center_y
                rm = rm_map[y, x]
                rm_err = rm_err_map[y, x]
                row_dict = {"stack":s, "x":delta_x, "y":delta_y, "rm":rm, "rm_err":rm_err}
                rows.append(row_dict)        

    #
    print("Done with this stack")
df_rm_stacks = pd.DataFrame(rows, columns=['stack', 'x', 'y', 'rm', 'rm_err'])
with open('df_rm_ang_stacks.pkl', 'wb') as f:
    pickle.dump(df_rm_stacks, f)

----- Extract RM values from individual stacks -----
Working on stack: stack_ang_align_noNorm_mean:
Done with this stack
Working on stack: stack_ang_align_noNorm_median:
Done with this stack
Working on stack: stack_ang_align_piNorm_mean:
Done with this stack
Working on stack: stack_ang_align_piNorm_mean_exclhighrm:
Done with this stack
Working on stack: stack_ang_align_piNorm_median:
Done with this stack
Working on stack: stack_ang_align_piNorm_median_exclhighrm:
Done with this stack
Working on stack: stack_ang_align_piNorm_median_highrm:
Done with this stack
Working on stack: stack_ang_align_piNorm_median_lowrm:
Done with this stack
Working on stack: stack_ang_standard_noNorm_mean:
Done with this stack
Working on stack: stack_ang_standard_noNorm_median:
Done with this stack
Working on stack: stack_ang_standard_piNorm_mean:
Done with this stack
Working on stack: stack_ang_standard_piNorm_mean_exclhighrm:
Done with this stack
Working on stack: stack_ang_standard_piNorm_median:
Done with

### Combining extracted RM values of individual galaxies

In [ ]:
def stack_rm(gal_df, rm_df, out_tag,
             align=False,
             rm_abs_norm=False):
    array_size = (rc.regrid_npix_ang, rc.regrid_npix_ang)
    rm_val = np.empty(array_size, dtype=object)
    rm_err = np.empty(array_size, dtype=object)
    n_rm = np.zeros(array_size, dtype=int)
    ref_head = fits.getheader("cube_processing/rm_analysis/3d_cubes/stack_ang_standard_noNorm_mean/pf_stack_ang_standard_noNorm_mean_phiPeakPIfit_rm2_pi_masked.fits")
    ref_wcs = WCS(ref_head)
    empty_head = ref_wcs.to_header()
    # Initialize each element in rm_val and rm_err as an empty list
    for idx, _ in np.ndenumerate(rm_val):
        rm_val[idx] = []
        rm_err[idx] = []

    # Calculate center
    center_x = array_size[0] // 2
    center_y = array_size[1] // 2

    weighted_mean = np.full(array_size, np.nan)
    unweighted_mean = np.full(array_size, np.nan)
    weighted_std = np.full(array_size, np.nan)
    median = np.full(array_size, np.nan)

    for index, row in gal_df.iterrows():
        gal = row['galaxy'].replace(" ","")
        df_rm_gal = rm_df[rm_df["gal"]==gal]
        print(f"Adding {gal} with {len(df_rm_gal)} pixels to the stack")
        extra_rot = row['extra_rot']
        if align:
            if extra_rot =="yes":
                print("Apply extra rotation")
                df_rm_gal['x'] = -1* df_rm_gal['x']
                df_rm_gal['y'] = -1* df_rm_gal['y']
        if rm_abs_norm:
            w_mean_abs_rm = row['abs_rm_weighted_mean']
            print(f"Normalising data using weighted mean of absolute RM values: {w_mean_abs_rm}")
            df_rm_gal['rm'] = df_rm_gal['rm'] *(rc.abs_rm_val_norm/w_mean_abs_rm)
            df_rm_gal['rm_err'] = df_rm_gal['rm_err'] *(rc.abs_rm_val_norm/w_mean_abs_rm)      
    # Iterate through the DataFrame
        for _, row_gal in df_rm_gal.iterrows():
            # Convert offset to array indices
            array_x = center_x + int(row_gal['x'])
            array_y = center_y + int(row_gal['y'])
            # Check bounds
            if 0 <= array_x < array_size[0] and 0 <= array_y < array_size[1]:
                rm_val[array_y, array_x].append(row_gal['rm'])
                rm_err[array_y, array_x].append(row_gal['rm_err'])
                n_rm[array_y, array_x] += 1
            else:
                print(f"Warning: ({array_x}, {array_y}) is out of bounds for array size {array_size}")


    for idx, _ in np.ndenumerate(rm_val):
        vals = rm_val[idx]
        errs = rm_err[idx]
        if len(vals) > 0 and all(e > 0 for e in errs):
            vals = np.array(vals)
            errs = np.array(errs)
            weights = 1.0 / errs**2
            mean = np.sum(weights * vals) / np.sum(weights)
            std = np.sqrt(np.sum(weights * (vals - mean)**2) / np.sum(weights))
            med = np.nanmedian(vals)
            mean_unweighted = np.nanmean(vals)
            weighted_mean[idx] = mean
            weighted_std[idx] = std
            median[idx]=med
            unweighted_mean[idx] = mean_unweighted
    count_positive_negative = np.zeros(array_size, dtype=int)

    for idx, vals in np.ndenumerate(rm_val):
        vals = np.array(vals)
        n_pos = np.sum(vals > 0)
        n_neg = np.sum(vals < 0)
        count_positive_negative[idx] = n_pos - n_neg
            
            
    mask = np.where(n_rm<rc.min_n_for_stack, np.nan, 1.)
    weighted_mean = weighted_mean*mask
    weighted_std = weighted_std*mask
    median = median*mask
    unweighted_mean = unweighted_mean*mask
    count_positive_negative = count_positive_negative*mask 
    header_rm = empty_head.copy()
    header_rm['BUNIT']="rad/m^2"
    header_rm["BMAJ"] = ref_head["BMAJ"]
    header_rm["BMIN"] = ref_head["BMIN"]              
    header_rm["BPA"] = ref_head["BPA"]              
    fits.writeto(f"cube_processing/stacks_rm_from_galaxies/{out_tag}_rm_wmean.fits",
                data=weighted_mean, header=header_rm, overwrite=True)
    fits.writeto(f"cube_processing/stacks_rm_from_galaxies/{out_tag}_rm_wstd.fits",
                data=weighted_std, header=header_rm, overwrite=True)
    fits.writeto(f"cube_processing/stacks_rm_from_galaxies/{out_tag}_rm_median.fits",
                data=median, header=header_rm, overwrite=True) 
    fits.writeto(f"cube_processing/stacks_rm_from_galaxies/{out_tag}_rm_unweighted_mean.fits",
                 data=unweighted_mean, header=header_rm, overwrite=True)      
    fits.writeto(f"cube_processing/stacks_rm_from_galaxies/{out_tag}_rm_n.fits",
                data=n_rm, header=empty_head, overwrite=True)
    fits.writeto(f"cube_processing/stacks_rm_from_galaxies/{out_tag}_rm_count_pos_neg.fits",
                data=count_positive_negative, header=empty_head, overwrite=True)
    # print statistics of individual quadrants
    npix = np.count_nonzero(~np.isnan(weighted_mean))
    print("Npix in stack:", npix)
    q1 = weighted_mean[:center_y,center_x:]
    q2 = weighted_mean[:center_y,:center_x]
    q3 = weighted_mean[center_y:, :center_x]
    q4 = weighted_mean[center_y:, center_x:]
    print("Quadrant Statistics (weighted mean RM):")
    q1q3 = np.concatenate((q1.ravel(), q3.ravel()))
    q2q4 = np.concatenate((q2.ravel(), q4.ravel()))
    # remove nan values
    q1q3 = q1q3[~np.isnan(q1q3)]
    q2q4 = q2q4[~np.isnan(q2q4)]
    # compute statistics
    q1q3_mean = np.mean(q1q3)
    q2q4_mean = np.mean(q2q4)
    q1q3_std = np.std(q1q3)
    q2q4_std = np.std(q2q4)
    nbeam_q1q3 = len(q1q3) / ((rc.ang_stack_pix_per_beam)**2)
    nbeam_q2q4 = len(q2q4) / ((rc.ang_stack_pix_per_beam)**2)
    q1q3_unc_mean = q1q3_std / np.sqrt(nbeam_q1q3)
    q2q4_unc_mean = q2q4_std / np.sqrt(nbeam_q2q4)
    delta_mu = q1q3_mean - q2q4_mean
    unc_delta_mu = np.sqrt(q1q3_unc_mean**2 + q2q4_unc_mean**2)
    
    print(f"Q1+Q3: mean={q1q3_mean}")
    print(f"Q2+Q4: mean={q2q4_mean}")
    print(f"Q1+Q3: std={q1q3_std}")
    print(f"Q2+Q4: std={q2q4_std})")
    print(f"Q1+Q3: Nbeam={nbeam_q1q3}")
    print(f"Q2+Q4: Nbeam={nbeam_q2q4}")
    print(f"Q1+Q3: Uncertainty on mean={q1q3_unc_mean}")
    print(f"Q2+Q4: Uncertainty on mean={q2q4_unc_mean}")
    print(f"Delta Mu (Q1+Q3 - Q2+Q4) = {delta_mu} +/- {unc_delta_mu} rad/m^2")
    print("LaTeX Table Row:")
    latex_row = (
        f"{npix:.0f} & " 
        f"${q1q3_mean:.0f} \\pm {q1q3_unc_mean:.0f}$ & "
        f"${q2q4_mean:.0f} \\pm {q2q4_unc_mean:.0f}$ & "
        f"${q1q3_std:.0f}$ & "
        f"${q2q4_std:.0f}$ & "
        f"${delta_mu:.0f} \\pm {unc_delta_mu:.0f}$ \\\\"
    )
    print(latex_row)
    
    
    

#### standard

In [ ]:
stack_rm(gal_df=df_ang,
         rm_df=df_rm,
         out_tag="stack_ang_standard")

In [ ]:
stack_rm(gal_df=df_ang,
         rm_df=df_rm,
         rm_abs_norm=True,
         out_tag="stack_ang_standard_RMnorm")

#### Align

In [ ]:
stack_rm(gal_df=df_ang,
         rm_df=df_rm,
         align=True,
         out_tag="stack_ang_align")


In [ ]:
stack_rm(gal_df=df_ang,
         rm_df=df_rm,
         align=True,
         rm_abs_norm=True,
         out_tag="stack_ang_align_RMnorm",
         )

## Prepare RM-Tools Input (NOT USED)

In [ ]:
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.wcs import WCS
from photutils import CircularAperture, aperture_photometry
from regions import Regions

import astropy.units as u


def extract_rm_tool_input(q_cube, u_cube, freq_file, reg_file, out_path, hdu=0, use_pixel_coords=False):
    # Load the q and u cubes that are stored in fits file format and also import the WCS information from the headers.
    q_data = fits.open(q_cube)[hdu].data
    u_data = fits.open(u_cube)[hdu].data
    q_header = fits.open(q_cube)[hdu].header
    u_header = fits.open(u_cube)[hdu].header
    w = WCS(q_header).celestial
    
    # Import the frequency information from the freq_file using np.loadtxt
    frequencies = np.loadtxt(freq_file)
    
    # Read the region file to get the mask
    regions = Regions.read(filename=reg_file, format='ds9')
    region = regions[0]
    
    if use_pixel_coords:
        reg_pix = region
    else:
        reg_pix = region.to_pixel(w)
    
    mask_source = reg_pix.to_mask().to_image([q_data.shape[1], q_data.shape[2]]).astype('bool')
    mask_bkg = np.invert(mask_source.astype('bool'))
    
    # Initialize lists to store the flux and error values
    q_mean_list = []
    u_mean_list = []
    q_err_list = []
    u_err_list = []
    
    # Loop through each spectral slice
    for i in range(q_data.shape[0]):
        q_slice = q_data[i, :, :]
        u_slice = u_data[i, :, :]
        q_slice_masked = q_slice * mask_bkg
        u_slice_masked = u_slice * mask_bkg
        
        q_mean = np.nanmean(q_slice_masked)
        u_mean = np.nanmean(u_slice_masked)
        q_err = np.nanstd(q_slice)
        u_err = np.nanstd(u_slice)
        
        # Append the values to the lists
        q_mean_list.append(q_mean)
        u_mean_list.append(u_mean)
        q_err_list.append(q_err)
        u_err_list.append(u_err)
        
    q_err_list=np.array(q_err_list)/1000
    u_err_list=np.array(u_err_list)/1000
    # Create a 5 column pandas data frame, storing frequency, q, u, q_err, u_err
    data = {
        'frequency': frequencies,
        'q': q_mean_list,
        'u': u_mean_list,
        'q_err': q_err_list,
        'u_err': u_err_list
    }
    df = pd.DataFrame(data)
    
    # Write the data frame to the out_path as an ASCII file
    df.to_csv(out_path, index=False, sep=' ', header=False)
    return df

extract_rm_tool_input(q_cube="/data/mstein/changes/qu_stack/rm_data/NGC891/RMsynth/qu_fits_C/Q.fits",
                      u_cube="/data/mstein/changes/qu_stack/rm_data/NGC891/RMsynth/qu_fits_C/U.fits",
                      freq_file="/data/mstein/changes/qu_stack/rm_data/NGC891/RMsynth/qu_fits_C/freq.txt",
                      reg_file="cube_processing/rm_analysis/apertures/n891_pi_peak_raw.reg",
                      out_path="cube_processing/rm_analysis/1d_spectra/n891_pi_peak_raw.dat",
                      )
# We can use the same region as in raw, because the cube has not ben rotated at this stage.
extract_rm_tool_input(q_cube="cube_processing/qu_source/NGC891_q_source.fits",
                      u_cube="cube_processing/qu_source/NGC891_u_source.fits",
                      freq_file="/data/mstein/changes/qu_stack/rm_data/NGC891/RMsynth/qu_fits_C/freq.txt",
                      reg_file="cube_processing/rm_analysis/apertures/n891_pi_peak_raw.reg",
                      out_path="cube_processing/rm_analysis/1d_spectra/n891_pi_peak_source.dat",
                      )
# Check what happens after galaxy alignment
extract_rm_tool_input(q_cube="cube_processing/qu_mask/NGC891_q_mask.fits",
                      u_cube="cube_processing/qu_mask/NGC891_u_mask.fits",
                      freq_file="/data/mstein/changes/qu_stack/rm_data/NGC891/RMsynth/qu_fits_C/freq.txt",
                      reg_file="cube_processing/rm_analysis/apertures/n891_pi_peak_processed.reg",
                      out_path="cube_processing/rm_analysis/1d_spectra/n891_pi_peak_mask.dat",
                      )
    

## Create Noise files

### Stacks

In [ ]:
def create_noise_file(stack_name, region_file, out_path, hdu=0,
                      head='cel', show_plots=False):
    print(f"Creating noise cube for {stack_name} using region file {region_file}")
    q_cube =f"cube_processing/q_{stack_name}.fits"
    u_cube =f"cube_processing/u_{stack_name}.fits"
    q_list=[]
    u_list=[]
    for cube, noise_list in zip([q_cube, u_cube], [q_list, u_list]):
        data = fits.open(cube)[hdu].data
        header = fits.open(cube)[hdu].header
        if head=='cel':
            w = WCS(header).celestial
            regions = Regions.read(filename=region_file, format='ds9')
            region = regions[0]
            reg_pix = region.to_pixel(w)
            mask_source = reg_pix.to_mask().to_image([data.shape[1], data.shape[2]]).astype('bool')

        if head=='phy':
            box_ap = RectangularAperture((54.23, 39.02),
                                         w=13.06,h=10.67, theta=0)
            mask_source = box_ap.to_mask().to_image([data.shape[1], data.shape[2]]).astype('bool')
        
            
        mask_source = np.where(mask_source, 1., np.nan)
        
        for i in range(data.shape[0]):
            slice_data = data[i, :, :]
            slice_data_masked = slice_data * mask_source
            if show_plots:
                plt.imshow(slice_data_masked)
                plt.colorbar()
                plt.show()
            noise_level = np.nanstd(slice_data_masked)
            noise_list.append(noise_level)
    print("Noise levels for Q:", q_list)    
    print("Noise levels for U:", u_list)    
    list_mean_noise = []
    for qn, un in zip(q_list, u_list):
        mean_noise = (qn + un)/2.
        list_mean_noise.append(mean_noise)
    print("Mean Noise levels:", list_mean_noise)
    np.savetxt(out_path, np.array(list_mean_noise))

In [ ]:
ang_stacks = [s for s in stacks if "stack_ang" in s]
phy_stacks = [s for s in stacks if "stack_phy" in s]

for stack in ang_stacks:
    create_noise_file(stack_name=stack,
                      region_file=f"stack_ang_backgorund_noise.reg",
                      out_path=f"/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/{stack}/noise.txt", head='cel')

for stack in phy_stacks:
    create_noise_file(stack_name=stack,
                      region_file=f"None",
                      out_path=f"/home/mstein/scripts/qu_stacking/cube_processing/rm_analysis/3d_cubes/{stack}/noise.txt", head='phy')


### Galaxies

In [ ]:
def create_noise_file_gals(gal_name, region_file, hdu=0,
                           head='cel', show_plots=False):
    print(f"Creating noise cube for {gal_name} using region file {region_file}")
    gal_no_space = gal_name.replace(" ","")
    if head=='cel':
        q_cube =f"cube_processing/qu_regrid_ang/{gal_no_space}_q_regrid_ang.fits"
        u_cube =f"cube_processing/qu_regrid_ang/{gal_no_space}_u_regrid_ang.fits"
        
    if head=='phy':
        q_cube =f"cube_processing/qu_regrid_phy/{gal_no_space}_q_regrid_phy.fits"
        u_cube =f"cube_processing/qu_regrid_phy/{gal_no_space}_u_regrid_phy.fits"
    q_list=[]
    u_list=[]
    for cube, noise_list in zip([q_cube, u_cube], [q_list, u_list]):
        data = fits.open(cube)[hdu].data
        header = fits.open(cube)[hdu].header
        if head=='cel':
            w = WCS(header).celestial
            regions = Regions.read(filename=region_file, format='ds9')
            region = regions[0]
            reg_pix = region.to_pixel(w)
            mask_source = reg_pix.to_mask().to_image([data.shape[1], data.shape[2]]).astype('bool')

        if head=='phy':
            box_ap = RectangularAperture((54.23, 39.02),
                                         w=13.06,h=10.67, theta=0)
            mask_source = box_ap.to_mask().to_image([data.shape[1], data.shape[2]]).astype('bool')
        
            
        mask_source = np.where(mask_source, 1., np.nan)
        
        for i in range(data.shape[0]):
            slice_data = data[i, :, :]
            slice_data_masked = slice_data * mask_source
            if show_plots:
                plt.imshow(slice_data_masked)
                plt.colorbar()
                plt.show()
            noise_level = np.nanstd(slice_data_masked)
            noise_list.append(noise_level)
    print("Noise levels for Q:", q_list)    
    print("Noise levels for U:", u_list)    
    list_mean_noise = []
    for qn, un in zip(q_list, u_list):
        mean_noise = (qn + un)/2.
        list_mean_noise.append(mean_noise)
    print("Mean Noise levels:", list_mean_noise)
    gal_id = gal_no_space.replace("NGC","n")
    print(gal_id)
    if head=='cel':
        np.savetxt(f"cube_processing/rm_analysis/3d_cubes/{gal_id}_regrid_ang/noise.txt", np.array(list_mean_noise))
    if head=='phy':
        np.savetxt(f"cube_processing/rm_analysis/3d_cubes/{gal_id}_regrid_phy/noise.txt", np.array(list_mean_noise))

In [ ]:
ang_gal = df_ang['galaxy'].tolist()
#ang_gal_key = [f"n{gal[3:]}".replace(' ','') for gal in ang_gal]

for gal in ang_gal:
    create_noise_file_gals(gal_name=gal, region_file=f"stack_ang_backgorund_noise.reg",head='cel')


In [ ]:
phy_gal = df_phy['galaxy'].tolist()
    
for gal in phy_gal:
    create_noise_file_gals(gal_name=gal, region_file=None,head='phy')